## Combining different results from all traffic assignments

Here, the different results from the traffic assignments for the baseline scenario as well as all outage scenarios simulated are combined into a single excel file.

In [16]:
import os
import pandas as pd
import numpy as np

def process_folder_system(folder_path, scenario_name):
    """Process a folder of CSV files and return combined DataFrame"""
    columns_to_extract = [
        'Origin Country', 'Num Links Used', 'Num Unique System_IDs', 'Avg VOC_max',
        'Avg Delay_factor_Max', 'Avg traffic_all', 'Uses Capital Link',
        'Num Capital Links Used', 'Capital Link Countries', 'Capital-to-Capital Traffic',
        'System_IDs', 'Links Used'
    ]
    list_dfs = []
    try:
        files = [
            f for f in os.listdir(folder_path)
            if f.startswith('final_df') and f.endswith('.csv')
        ]
        for file in files:
            file_path = os.path.join(folder_path, file)
            try:
                df = pd.read_csv(file_path, usecols=columns_to_extract)
                outage_of = file.split('_')[-1].replace('.csv', '')
                df['Scenario'] = scenario_name
                df['Outage of'] = outage_of
                list_dfs.append(df)
            except Exception as e:
                print(f"Error reading {file}: {e}")
    except (FileNotFoundError, NotADirectoryError) as e:
        print(f"Error accessing folder: {folder_path} - {str(e)}")
    if list_dfs:
        return pd.concat(list_dfs, ignore_index=True)
    else:
        return pd.DataFrame(columns=columns_to_extract + ['Scenario', 'Outage of'])

def process_folder_ownership_and_cluster(folder_path, scenario_name):
    """Process a folder of CSV files and return combined DataFrame"""
    columns_to_extract = [
        'Origin Country', 'Num Links Used', 'Num Unique System_IDs', 'Avg VOC_max',
        'Avg Delay_factor_Max', 'Avg traffic_all', 'Uses Capital Link',
        'Num Capital Links Used', 'Capital Link Countries', 'Capital-to-Capital Traffic',
        'System_IDs', 'Links Used'
    ]
    list_dfs = []
    try:
        files = [
            f for f in os.listdir(folder_path)
            if f.startswith('final_df') and f.endswith('.csv')
        ]
        for file in files:
            file_path = os.path.join(folder_path, file)
            try:
                df = pd.read_csv(file_path, usecols=columns_to_extract)
                if 'removed_' in file:
                    outage_of = file.split('removed_')[1].split('.')[0]
                else:
                    outage_of = file.split('_')[-1].split('.')[0]  # fallback
                df['Scenario'] = scenario_name
                df['Outage of'] = outage_of
                list_dfs.append(df)
            except Exception as e:
                print(f"Error reading {file}: {e}")
    except (FileNotFoundError, NotADirectoryError) as e:
        print(f"Error accessing folder: {folder_path} - {str(e)}")
    if list_dfs:
        return pd.concat(list_dfs, ignore_index=True)
    else:
        return pd.DataFrame(columns=columns_to_extract + ['Scenario', 'Outage of'])

def process_baseline_file(file_path):
    """Process baseline file and return DataFrame"""
    columns_to_extract = [
        'Origin Country', 'Num Links Used', 'Num Unique System_IDs', 'Avg VOC_max',
        'Avg Delay_factor_Max', 'Avg traffic_all', 'Uses Capital Link',
        'Num Capital Links Used', 'Capital Link Countries', 'Capital-to-Capital Traffic',
        'System_IDs', 'Links Used'
    ]
    try:
        df = pd.read_csv(file_path)
        # Select only the required columns (in case extra columns exist)
        df = df[columns_to_extract].copy()
        df['Scenario'] = 'baseline'
        df['Outage of'] = 'baseline'
        return df
    except FileNotFoundError:
        print(f"Baseline file not found: {file_path}")
        return pd.DataFrame(columns=columns_to_extract + ['Scenario', 'Outage of'])
    except Exception as e:
        print(f"Error reading baseline file: {str(e)}")
        return pd.DataFrame(columns=columns_to_extract + ['Scenario', 'Outage of'])

# Process system outage scenarios
system_folder = r'C:\Users\Dean\Downloads\IfW\scenario_analysis_results'
system_df = process_folder_system(system_folder, 'System Outage')

# Process ownership outage scenarios
ownership_folder = r'C:\Users\Dean\Downloads\IfW\ownership_analysis_results'
ownership_df = process_folder_ownership_and_cluster(ownership_folder, 'Ownership Outage')

# Process cluster outage scenarios
cluster_folder = r'C:\Users\Dean\Downloads\IfW\cluster_analysis_results'
cluster_df = process_folder_ownership_and_cluster(cluster_folder, 'Cluster Outage')

# Process baseline file
baseline_file = r'C:\Users\Dean\Downloads\IfW\baselineTA.csv'
baseline_df = process_baseline_file(baseline_file)

# Combine all DataFrames
dfs_to_combine = [system_df, ownership_df, cluster_df, baseline_df]
combined_df = pd.concat([df for df in dfs_to_combine if not df.empty], ignore_index=True)

# Add "Number of outages" column with default values
if not combined_df.empty:
    combined_df['Granularity'] = combined_df['Origin Country'].apply(
        lambda x: 'Global' if x == 'Average' else 'Country'
    )
    
    # Initialize with default values
    combined_df['Number of outages'] = combined_df['Scenario'].apply(
        lambda x: 1 if x == 'System Outage' else 0
    )

# Step 1: Read ownership_systems_removed.xlsx
try:
    ownership_systems_removed = pd.read_excel('ownership_systems_removed.xlsx')
    print("Successfully loaded ownership_systems_removed.xlsx")
except FileNotFoundError:
    print("Error: ownership_systems_removed.xlsx not found. Using empty DataFrame.")
    ownership_systems_removed = pd.DataFrame(columns=['Affiliation', 'Unique System IDs Removed'])

# Step 2: Update 'Number of outages' for Ownership Outage scenarios
if not combined_df.empty and not ownership_systems_removed.empty:
    # Replace underscores with spaces for Ownership Outage rows in 'Outage of'
    ownership_mask = combined_df['Scenario'] == 'Ownership Outage'
    combined_df.loc[ownership_mask, 'Outage of'] = (
        combined_df.loc[ownership_mask, 'Outage of'].str.replace('_', ' ')
    )
    
    # Create mapping dictionary (Affiliation uses spaces)
    outage_mapping = ownership_systems_removed.set_index('Affiliation')['Unique System IDs Removed'].to_dict()
    
    # Update only Ownership Outage rows
    combined_df.loc[ownership_mask, 'Number of outages'] = (
        combined_df.loc[ownership_mask, 'Outage of'].map(outage_mapping).fillna(0)
    )
    
    # Convert to integer
    combined_df['Number of outages'] = combined_df['Number of outages'].astype(int)

# Add Average distance travelled per link column
if not combined_df.empty:
    # Load the edges data
    edges_file = r'C:\Users\Dean\Downloads\IfW\2024FullNetworkEdges.csv'
    try:
        edges_df = pd.read_csv(edges_file)
        print("Successfully loaded edges data")
        
        # Create mapping from index to distance_km (for speed)
        edges_distance = edges_df['distance_km'].to_dict()
        # Also collect all valid link indices
        valid_link_indices = set(edges_df.index)
        
        # Create function to calculate average distance with fallback
        def calculate_avg_distance(row):
            if row['Granularity'] != 'Country':
                return np.nan
            links_str = row['Links Used']
            if pd.isna(links_str) or links_str.strip() == '':
                return np.nan
            # Parse the links used into a list of integers
            link_indices = [int(x.strip()) for x in links_str.split(',') if x.strip().isdigit()]
            # For each link, get its distance (50 if not found)
            distances = []
            for idx in link_indices:
                if idx in valid_link_indices:
                    distances.append(edges_distance.get(idx, 50))  # Should be redundant, but safe
                else:
                    distances.append(50)
            if len(distances) == 0 or row['Num Links Used'] == 0:
                return np.nan
            avg_distance = sum(distances) / row['Num Links Used']
            return avg_distance
        
        # Apply the function to combined_df
        combined_df['Average distance travelled per link'] = combined_df.apply(calculate_avg_distance, axis=1)
        
    except FileNotFoundError:
        print(f"Error: Edges file not found at {edges_file}")
        combined_df['Average distance travelled per link'] = np.nan
    except Exception as e:
        print(f"Error processing edges data: {str(e)}")
        combined_df['Average distance travelled per link'] = np.nan
else:
    print("No data to process")

# --- Now calculate the adjusted average for "Average" rows ---
# Add the column if not present
if 'Adjusted Average distance travelled per link' not in combined_df.columns:
    combined_df['Adjusted Average distance travelled per link'] = combined_df['Average distance travelled per link']

# Find indices where Origin Country is 'Average'
average_indices = combined_df.index[combined_df['Origin Country'] == 'Average'].tolist()

# Iterate over the segments between 'Average' rows
for i in range(len(average_indices)):
    start_idx = average_indices[i]
    end_idx = average_indices[i+1] if i+1 < len(average_indices) else len(combined_df)
    
    # Calculate mean of 'Average distance travelled per link' for the segment above
    if i == 0:
        segment = combined_df.loc[:end_idx-1, 'Average distance travelled per link']
    else:
        prev_avg_idx = average_indices[i-1]
        segment = combined_df.loc[prev_avg_idx+1:end_idx-1, 'Average distance travelled per link']
    segment_mean = segment.mean()
    
    # Assign this mean to the 'Average' row at start_idx
    combined_df.at[start_idx, 'Adjusted Average distance travelled per link'] = segment_mean


# Show final results
print(f"Total rows: {len(combined_df)}")
if not combined_df.empty:
    combined_df
else:
    print("No data found in any folders or files")


Successfully loaded ownership_systems_removed.xlsx
Successfully loaded edges data
Total rows: 46627


In [17]:
combined_df

,Origin Country,Num Links Used,Num Unique System_IDs,Avg VOC_max,Avg Delay_factor_Max,Avg traffic_all,Uses Capital Link,Num Capital Links Used,Capital Link Countries,Capital-to-Capital Traffic,System_IDs,Links Used,Scenario,Outage of,Granularity,Number of outages,Average distance travelled per link,Adjusted Average distance travelled per link
0,Albania,6.000000,3.000000,0.028887,1.004333,784.000000,False,0.0,NaN,0.0,"2000.3, 2023.29, nan","58, 1182, 1374, 1375, 2427, 2428",System Outage,2000.1,Country,1,39.423209,39.423209
1,Algeria,13.000000,4.000000,0.190311,1.028547,5161.563248,False,0.0,NaN,0.0,"2002.7, 2005.3, 2005.8, nan","244, 286, 295, 296, 297, 298, 299, 1538, 1539,...",System Outage,2000.1,Country,1,188.134288,188.134288
2,Angola,30.000000,5.000000,0.143431,1.021515,3892.715741,False,0.0,NaN,0.0,"2010.1, 2012.15, 2018.15, 2023.2, nan","511, 512, 643, 647, 648, 649, 650, 658, 659, 6...",System Outage,2000.1,Country,1,758.491463,758.491463
3,Argentina,19.000000,7.000000,0.217228,1.032584,5895.578947,False,0.0,NaN,0.0,"2000.27, 2010.15, 2011.13, 2018.9, 2023.1, 202...","51, 519, 587, 932, 1140, 1141, 1142, 1364, 136...",System Outage,2000.1,Country,1,923.217562,923.217562
4,Australia,61.000000,12.000000,0.264992,1.039749,7191.885428,False,0.0,NaN,0.0,"2000.33, 2006.7, 2017.11, 2018.12, 2018.14, 20...","62, 65, 66, 67, 68, 70, 72, 73, 74, 75, 76, 77...",System Outage,2000.1,Country,1,644.733283,644.733283
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46622,Uruguay,14.000000,5.000000,0.177515,1.026627,4817.744286,False,0.0,NaN,0.0,"2010.15, 2011.13, 2018.1, 2023.1, nan","521, 589, 905, 1142, 1143, 1840, 1841, 1897, 1...",baseline,baseline,Country,0,333.270614,333.270614
46623,Venezuela,24.000000,6.000000,0.266159,1.039924,7223.543333,False,0.0,NaN,0.0,"2000.13, 2000.14, 2001.11, 2001.15, 2011.5, nan","8, 9, 114, 115, 116, 117, 125, 131, 132, 133, ...",baseline,baseline,Country,0,491.362591,491.362591
46624,Vietnam,54.000000,7.000000,0.826526,1.123979,22431.923700,False,0.0,NaN,0.0,"2009.11, 2009.12, 2009.7, 2016.1, 2022.5, 2022...","448, 449, 450, 451, 452, 453, 456, 458, 461, 4...",baseline,baseline,Country,0,552.125705,552.125705
46625,Yemen,27.000000,3.000000,0.177293,1.026594,4801.285741,False,0.0,NaN,0.0,"2006.17, 2017.16, nan","348, 350, 351, 354, 355, 364, 366, 367, 865, 8...",baseline,baseline,Country,0,445.271350,445.271350


In [18]:
# Define output file path
output_excel_path = r'C:\Users\Dean\Downloads\IfW\combined_TA_results_all_scenarios3.xlsx'

# Save DataFrame to Excel
combined_df.to_excel(output_excel_path, index=False)

print(f"Combined results saved to: {output_excel_path}")

Combined results saved to: C:\Users\Dean\Downloads\IfW\combined_TA_results_all_scenarios3.xlsx
